# Commons Asset Downloader

Downloads the actual media files referenced in the catalog spreadsheets (images
and/or animations) onto your own computer, sorted into folders by category.

**How to use this notebook:**
1. Run the **Setup** cell once to make sure `openpyxl` and `requests` are installed.
2. Edit the values in the **Configuration** cell below (which spreadsheet(s) to
   read, where to save files, etc).
3. Run the **Functions** cell (defines everything, no output expected).
4. Run the **Download** cell to actually fetch the files.
5. Check the **Summary** cell at the end for a results table.

Re-running the Download cell is safe — files already on disk are skipped.

## Setup
Run this once. If the packages are already installed, it will just confirm that and do nothing else.

In [ ]:
import sys
import subprocess

for pkg in ("openpyxl", "requests"):
    try:
        __import__(pkg)
        print(f"'{pkg}' already installed.")
    except ImportError:
        print(f"Installing '{pkg}'...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])


## Configuration

Edit these values, then run this cell.

- `CATALOGS`: path(s) to the `.xlsx` catalog file(s). Add both if you want to download images and animations in one pass.
- `OUTPUT_FOLDER`: where downloaded files will be saved (created if it doesn't exist).
- `DRY_RUN`: `True` to preview without saving anything; `False` to actually download.
- `MAX_SIZE_MB`: skip files larger than this many MB (`None` = no limit). Useful for the animation catalog, which has some large videos.
- `THROTTLE_SECONDS`: pause between downloads, to be polite to Wikimedia's servers.

In [ ]:
CATALOGS = [
    "Public_Domain_Math_Assets_Catalog.xlsx",
    # "Public_Domain_Math_Animations_Catalog.xlsx",
]

OUTPUT_FOLDER = "downloads"

DRY_RUN = True          # set to False once you've confirmed the dry run looks right

MAX_SIZE_MB = None       # e.g. 50 to skip anything over 50 MB

THROTTLE_SECONDS = 1.0


## Functions
Run this cell to define everything. No output expected.

In [ ]:
import csv
import re
import time
import urllib.parse
from pathlib import Path
from typing import Iterator, Optional, Tuple

import requests
from openpyxl import load_workbook

# Wikimedia asks bots/scripts to identify themselves with a descriptive
# User-Agent. Feel free to edit the contact info below.
USER_AGENT = (
    "MathAssetDownloaderNotebook/1.0 "
    "(personal/educational use; contact: replace-with-your-email@example.com) "
    "python-requests"
)

REQUEST_TIMEOUT = 30      # seconds
RETRY_COUNT = 3
RETRY_DELAY = 3           # seconds between retries


def extract_filename_from_commons_url(url: str) -> str:
    """
    Given a Wikimedia Commons file page URL like
    'https://commons.wikimedia.org/wiki/File:Fractal_Broccoli.jpg',
    return the decoded file name 'Fractal Broccoli.jpg'.
    """
    marker = "/wiki/File:"
    idx = url.find(marker)
    if idx == -1:
        raise ValueError(f"Not a recognizable Commons File: URL: {url}")
    raw_name = url[idx + len(marker):]
    raw_name = urllib.parse.unquote(raw_name)
    raw_name = raw_name.replace("_", " ")
    return raw_name


def special_filepath_url(filename: str) -> str:
    """
    Build the Special:FilePath URL for a Commons file name. Requesting this
    URL 302-redirects to the actual file on upload.wikimedia.org, so it works
    as a generic "give me the current version of this file" endpoint.
    """
    return "https://commons.wikimedia.org/wiki/Special:FilePath/" + urllib.parse.quote(filename)


def safe_filename(name: str) -> str:
    """Strip characters that are illegal in Windows/Mac/Linux file names."""
    cleaned = re.sub(r'[<>:"/\\\\|?*]', "_", name).strip()
    return cleaned or "unnamed_file"


def find_column(header_row, *keywords) -> Optional[int]:
    """Return the index of the first header cell containing all keywords (case-insensitive)."""
    for i, cell in enumerate(header_row):
        text = str(cell or "").strip().lower()
        if all(k in text for k in keywords):
            return i
    return None


def read_catalog(xlsx_path: Path) -> Iterator[Tuple[str, str, str]]:
    """Yield (category, file_name, commons_url) tuples from a catalog spreadsheet."""
    wb = load_workbook(xlsx_path, read_only=True, data_only=True)
    ws = wb.active
    rows = list(ws.iter_rows(values_only=True))
    wb.close()

    if not rows:
        return

    header = rows[0]
    cat_idx = find_column(header, "categ")
    name_idx = find_column(header, "file", "name")
    url_idx = find_column(header, "url")

    if url_idx is None:
        print(f"  ! Could not find a URL column in {xlsx_path.name} - skipping this file.")
        return

    for row in rows[1:]:
        if not row or all(v is None for v in row):
            continue
        category = str(row[cat_idx]).strip() if cat_idx is not None and row[cat_idx] else "Uncategorized"
        file_name = str(row[name_idx]).strip() if name_idx is not None and row[name_idx] else ""
        url = str(row[url_idx]).strip() if row[url_idx] else ""
        if url:
            yield category, file_name, url


def remote_size_mb(session: requests.Session, file_url: str) -> Optional[float]:
    """Best-effort HEAD request to check file size before downloading. Returns None if unknown."""
    try:
        resp = session.head(file_url, timeout=REQUEST_TIMEOUT, allow_redirects=True)
        length = resp.headers.get("Content-Length")
        if length:
            return int(length) / (1024 * 1024)
    except requests.RequestException:
        pass
    return None


def download_file(
    session: requests.Session,
    commons_url: str,
    dest_path: Path,
    max_size_mb: Optional[float],
    dry_run: bool,
) -> Tuple[str, str]:
    """
    Download one file via Special:FilePath, with retries.
    Returns (status, message) where status is one of:
    'ok', 'skipped', 'too_large', 'error'
    """
    if dest_path.exists() and dest_path.stat().st_size > 0:
        return "skipped", "already exists on disk"

    try:
        filename = extract_filename_from_commons_url(commons_url)
    except ValueError as e:
        return "error", str(e)

    file_url = special_filepath_url(filename)

    if max_size_mb is not None:
        size = remote_size_mb(session, file_url)
        if size is not None and size > max_size_mb:
            return "too_large", f"{size:.1f} MB > limit of {max_size_mb} MB"

    if dry_run:
        return "ok", "would download (dry run)"

    last_error = ""
    for attempt in range(1, RETRY_COUNT + 1):
        try:
            resp = session.get(file_url, timeout=REQUEST_TIMEOUT, stream=True, allow_redirects=True)
            if resp.status_code == 200:
                dest_path.parent.mkdir(parents=True, exist_ok=True)
                tmp_path = dest_path.with_suffix(dest_path.suffix + ".part")
                with open(tmp_path, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=1024 * 256):
                        if chunk:
                            f.write(chunk)
                tmp_path.rename(dest_path)
                return "ok", "downloaded"
            elif resp.status_code == 404:
                return "error", f"404 not found ({file_url})"
            else:
                last_error = f"HTTP {resp.status_code}"
        except requests.RequestException as e:
            last_error = str(e)
        if attempt < RETRY_COUNT:
            time.sleep(RETRY_DELAY)

    return "error", f"failed after {RETRY_COUNT} attempts: {last_error}"

print("Functions defined.")


## Download
This runs through every catalog in `CATALOGS` and downloads each file. Progress prints below as it goes.

In [ ]:
out_root = Path(OUTPUT_FOLDER)
out_root.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

log_rows = []
counts = {"ok": 0, "skipped": 0, "too_large": 0, "error": 0}
total = 0

for catalog_arg in CATALOGS:
    catalog_path = Path(catalog_arg)
    if not catalog_path.exists():
        print(f"! Skipping missing file: {catalog_path}")
        continue

    print(f"\n=== Reading {catalog_path.name} ===")
    entries = list(read_catalog(catalog_path))
    print(f"  Found {len(entries)} entries")

    for category, file_name, commons_url in entries:
        total += 1
        cat_folder = safe_filename(category)
        base_name = safe_filename(file_name) if file_name else safe_filename(
            extract_filename_from_commons_url(commons_url)
        )
        dest = out_root / cat_folder / base_name

        print(f"  [{total}] {cat_folder}/{base_name} ...", end=" ")
        status, msg = download_file(session, commons_url, dest, MAX_SIZE_MB, DRY_RUN)
        print(f"{status.upper()} - {msg}")

        counts[status] = counts.get(status, 0) + 1
        log_rows.append({
            "catalog": catalog_path.name,
            "category": category,
            "file_name": base_name,
            "commons_url": commons_url,
            "status": status,
            "detail": msg,
        })

        if status == "ok" and not DRY_RUN:
            time.sleep(THROTTLE_SECONDS)

log_path = out_root / "_download_log.csv"
with open(log_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f, fieldnames=["catalog", "category", "file_name", "commons_url", "status", "detail"]
    )
    writer.writeheader()
    writer.writerows(log_rows)

print("\nDone. See the Summary cell below, and the log at:", log_path)


## Summary

In [ ]:
print("=" * 50)
print(f"Total entries processed : {total}")
print(f"  downloaded            : {counts.get('ok', 0)}")
print(f"  skipped (on disk)      : {counts.get('skipped', 0)}")
print(f"  too large              : {counts.get('too_large', 0)}")
print(f"  errors                 : {counts.get('error', 0)}")

if DRY_RUN:
    print("\nThis was a DRY RUN - no files were actually saved.")
    print("Set DRY_RUN = False in the Configuration cell above and re-run to download for real.")

if counts.get("error", 0):
    print("\nSome entries errored out - check the 'detail' column in the log CSV,")
    print("or inspect log_rows below, for specifics (a common cause is a file")
    print("being renamed or deleted on Commons since the catalog was built).")
    errors = [r for r in log_rows if r["status"] == "error"]
    for r in errors[:10]:
        print(f"  - {r['file_name']}: {r['detail']}")
